# 1_Baseline_Experiments

### Step 1: Import libraries and setup paths
This cell imports all required Python libraries for data processing, machine learning, and evaluation.  
It also includes scikit-learn components for preprocessing, modeling, and scoring.

In [1]:
import pandas as pd
from pathlib import Path
import time, os, json
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

### Step 2: Define dataset configuration
Here we define the paths and basic settings for all datasets used in this exercise. Each entry includes file paths, target column, and separators. We use four datasets (two from Kaggle and two from UCI).


In [2]:
DATA_BASE = Path("../data")
RESULTS_DIR = Path("../results"); RESULTS_DIR.mkdir(exist_ok=True, parents=True)

# config for ALL FOUR datasets
CFG = {
    "wine": {
        "type": "uci_wine",
        "red":   DATA_BASE / "WineQuality" / "winequality-red.csv",
        "white": DATA_BASE / "WineQuality" / "winequality-white.csv",
        "sep":   ";",
        "target":"quality",
    },
    "adult": {
        "type": "uci_adult",
        "path": DATA_BASE / "Adult" / "adult.data",
        "sep":  ",",
        "target":"income",
        "na_values": [" ?"],
    },
    "cancer": {
        "type": "kaggle_lrn",
        "path": DATA_BASE / "cancer" / "breast-cancer-diagnostic.shuf.lrn.csv",
        "sep":  ",",
        "target":"class",
    },
    "loan": {
        "type": "kaggle_lrn",
        "path": DATA_BASE / "loan" / "loan-10k.lrn.csv",
        "sep":  ",",
        "target":"grade",
    },
}

### Step 3: Define preprocessing pipeline
This function builds a unified preprocessing pipeline that:
- imputes missing numeric values with the median
- imputes categorical values with the most frequent class
- scales numerical features (StandardScaler)
- encodes categorical variables (OneHotEncoder)

Using ColumnTransformer ensures consistent preprocessing across datasets and models.


In [3]:
# Preprocessor (single source of truth) 
def build_preprocessor(X: pd.DataFrame):
    num_cols = X.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    num_tr = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    cat_tr = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", num_tr, num_cols),
        ("cat", cat_tr, cat_cols)
    ])

### Step 4: Define machine learning models
We define three classifiers from different algorithmic families:
- Logistic Regression (linear model)
- Random Forest (ensemble, tree-based)
- Support Vector Machine with RBF kernel (non-linear)


In [4]:
MODELS = {
    "Logistic Regression": LogisticRegression(max_iter=300, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "SVM (RBF)": SVC(kernel="rbf", class_weight="balanced"),
}

LOGREG_GRID = [{"clf__C": c} for c in [0.1, 1.0, 10]]
RF_GRID     = [{"clf__n_estimators": n, "clf__max_depth": d} 
               for n in [200, 300] for d in [None, 20]]
SVM_GRID    = [{"clf__C": c, "clf__gamma": g} 
               for c in [0.5, 1.0, 2.0] for g in ["scale", "auto"]]

GRID_BY_MODEL = {
    "Logistic Regression": LOGREG_GRID,
    "Random Forest": RF_GRID,
    "SVM (RBF)": SVM_GRID,
}

### Step 5: Dataset loading utility
This function loads and prepares datasets based on their configuration type.
It handles:
- Reading CSVs (with separators and missing value indicators)
- Combining multiple files (red & white wine)
- Extracting features (X) and target (y)


In [5]:
# loaders
def load_dataset(name: str, cfg: dict):
    t = cfg["type"]
    if t == "uci_wine":
        red = pd.read_csv(cfg["red"], sep=cfg["sep"])
        white = pd.read_csv(cfg["white"], sep=cfg["sep"])
        df = pd.concat([red, white], ignore_index=True)
        X, y = df.drop(columns=[cfg["target"]]), df[cfg["target"]].astype("category")
        return X, y
    elif t == "uci_adult":
        cols = ["age","workclass","fnlwgt","education","education-num","marital-status",
                "occupation","relationship","race","sex","capital-gain","capital-loss",
                "hours-per-week","native-country","income"]
        df = pd.read_csv(cfg["path"], names=cols, sep=cfg["sep"],
                         na_values=cfg.get("na_values"), skipinitialspace=True)
        X, y = df.drop(columns=[cfg["target"]]), df[cfg["target"]].astype("category")
        return X, y
    elif t == "kaggle_lrn":
        df = pd.read_csv(cfg["path"], sep=cfg["sep"])
        X, y = df.drop(columns=[cfg["target"]]), df[cfg["target"]].astype("category")
        return X, y
    else:
        raise ValueError(f"Unknown dataset type: {t}")

### Step 6: Safe train/test split
Performs a stratified train-test split if all classes have at least two samples. If not, it falls back to a simple random split.


In [6]:
# safe split (if any class has <2 samples, drop stratify)
def safe_split(X, y, test_size=0.3, seed=42):
    vc = pd.Series(y).value_counts()
    stratify = y if vc.min() >= 2 else None
    return train_test_split(X, y, test_size=test_size, random_state=seed, stratify=stratify)

### Step 7: Run baseline experiments
This function:
- Loads each dataset
- Builds a preprocessing + classifier pipeline
- Trains and evaluates models on a holdout split
- Records accuracy, balanced accuracy, macro F1, and runtime

Results are printed and saved to CSV for later comparison.


In [7]:
# runner
def run_one(name: str, cfg: dict):
    print(f"\n=== {name.upper()} ===")
    X, y = load_dataset(name, cfg)
    X_tr, X_te, y_tr, y_te = safe_split(X, y, test_size=0.30, seed=42)
    pre = build_preprocessor(X)

    rows = []
    for mname, model in MODELS.items():
        pipe = Pipeline([("pre", pre), ("clf", model)])
        t0 = time.time(); pipe.fit(X_tr, y_tr); rt = time.time() - t0
        pred = pipe.predict(X_te)
        rows.append({
            "Model": mname,
            "Accuracy": accuracy_score(y_te, pred),
            "BalancedAcc": balanced_accuracy_score(y_te, pred),
            "MacroF1": f1_score(y_te, pred, average="macro"),
            "Runtime": rt
        })
        print(f"{mname}: acc={rows[-1]['Accuracy']:.3f}, bal_acc={rows[-1]['BalancedAcc']:.3f}, "
              f"f1={rows[-1]['MacroF1']:.3f}, time={rt:.2f}s")

    out = pd.DataFrame(rows)
    out_path = RESULTS_DIR / f"{name}_baseline.csv"
    out.to_csv(out_path, index=False)
    print("Saved")
    return out

### Step 8: Execute all baseline runs
Iterates over all dataset configurations and executes `run_one()` for each. Saves the baseline evaluation results (one CSV per dataset).


In [8]:
# run ALL (uses keys present in CFG)
for name, cfg in CFG.items():
    run_one(name, cfg)


=== WINE ===
Logistic Regression: acc=0.298, bal_acc=0.275, f1=0.195, time=0.37s
Random Forest: acc=0.677, bal_acc=0.364, f1=0.401, time=8.41s
SVM (RBF): acc=0.418, bal_acc=0.355, f1=0.263, time=4.54s
Saved

=== ADULT ===
Logistic Regression: acc=0.809, bal_acc=0.825, f1=0.774, time=0.77s
Random Forest: acc=0.856, bal_acc=0.780, f1=0.793, time=139.43s
SVM (RBF): acc=0.803, bal_acc=0.830, f1=0.770, time=75.82s
Saved

=== CANCER ===
Logistic Regression: acc=0.977, bal_acc=0.975, f1=0.975, time=0.03s
Random Forest: acc=0.907, bal_acc=0.907, f1=0.903, time=1.39s
SVM (RBF): acc=0.930, bal_acc=0.926, f1=0.926, time=0.04s
Saved

=== LOAN ===


C:\Users\canta\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression: acc=0.854, bal_acc=0.749, f1=0.722, time=2.57s
Random Forest: acc=0.825, bal_acc=0.513, f1=0.508, time=22.89s
SVM (RBF): acc=0.785, bal_acc=0.622, f1=0.633, time=18.24s
Saved


### Step 9: Measure runtime (fit vs predict)
This section separately measures training and prediction times. It logs metrics and timing data per (dataset × model) combination. The output is stored in `results/runtime.csv` for later analysis.


In [9]:
# Detailed execution time measurement / Separation of fit and predict
RUNTIME_OUT = RESULTS_DIR / "runtime.csv" 

def record_runtime_row(pipe, X_tr, y_tr, X_te, y_te, meta):
    t0 = time.perf_counter(); pipe.fit(X_tr, y_tr); fit_t = time.perf_counter() - t0
    t1 = time.perf_counter(); y_pred = pipe.predict(X_te); pred_t = time.perf_counter() - t1

    from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
    row = dict(
        **meta,
        n_train=len(X_tr), n_test=len(X_te), n_features=X_tr.shape[1],
        fit_time_sec=fit_t, pred_time_sec=pred_t,
        acc=accuracy_score(y_te, y_pred),
        bal_acc=balanced_accuracy_score(y_te, y_pred),
        f1_macro=f1_score(y_te, y_pred, average="macro"),
    )
    return row

rows = []
for ds_name, cfg in CFG.items():
    X, y = load_dataset(ds_name, cfg)
    X_tr, X_te, y_tr, y_te = safe_split(X, y, test_size=0.30, seed=42)
    pre = build_preprocessor(X)
    for mname, model in MODELS.items():
        pipe = Pipeline([("pre", pre), ("clf", model)])
        meta = {"dataset": ds_name, "model": mname, "preproc": "baseline", "split": "holdout"}
        rows.append(record_runtime_row(pipe, X_tr, y_tr, X_te, y_te, meta))

os.makedirs(RESULTS_DIR, exist_ok=True)
pd.DataFrame(rows).to_csv(RUNTIME_OUT, mode="a", header=not RUNTIME_OUT.exists(), index=False)
print(f"Appended {len(rows)} rows to {RUNTIME_OUT}")

C:\Users\canta\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Appended 12 rows to ..\results\runtime.csv


### Step 10: Evaluate models with cross-validation
Performs 5-fold Stratified CV to get more reliable estimates of model performance. Calculates mean and std of accuracy, balanced accuracy, and macro-F1 across folds. Results are appended to `results/cv.csv`.


In [10]:
# Cross-Validation (CV)
CV_OUT = RESULTS_DIR / "cv.csv"

def cv_eval_row(pipe, X, y, meta, n_splits=5):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scorers = {"acc": "accuracy", "bal_acc": "balanced_accuracy", "f1_macro": "f1_macro"}
    out = cross_validate(pipe, X, y, cv=cv, scoring=scorers, n_jobs=-1, return_train_score=False)
    row = dict(
        **meta,
        cv_splits=n_splits,
        acc_mean=out["test_acc"].mean(),       acc_std=out["test_acc"].std(),
        bal_acc_mean=out["test_bal_acc"].mean(), bal_acc_std=out["test_bal_acc"].std(),
        f1_macro_mean=out["test_f1_macro"].mean(), f1_macro_std=out["test_f1_macro"].std(),
    )
    return row

rows = []
for ds_name, cfg in CFG.items():
    X, y = load_dataset(ds_name, cfg)
    pre = build_preprocessor(X)
    for mname, model in MODELS.items():
        pipe = Pipeline([("pre", pre), ("clf", model)])
        meta = {"dataset": ds_name, "model": mname, "preproc": "baseline", "split": "cv"}
        rows.append(cv_eval_row(pipe, X, y, meta, n_splits=5))

os.makedirs(RESULTS_DIR, exist_ok=True)
pd.DataFrame(rows).to_csv(CV_OUT, mode="a", header=not CV_OUT.exists(), index=False)
print(f"Appended {len(rows)} rows to {CV_OUT}")

Appended 12 rows to ..\results\cv.csv


### Step 11: Ablation study — Scaling ON vs OFF for SVM
Tests the impact of feature scaling on SVM performance.  
Runs two CV experiments per dataset:
- **scale_on**: StandardScaler included  
- **scale_off**: No scaling on numeric features  
Results are saved to `results/ablation.csv` for direct comparison.


In [11]:
# Preprocessing Ablation (Scaling ON vs OFF for SVM)
ABL_OUT = RESULTS_DIR / "ablation.csv"

def build_preprocessor_no_scaler(X: pd.DataFrame):
    # Same column extraction and missing/OHE handling as the existing build_preprocessor, but removes the StandardScaler for numerical data.
    num_cols = X.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    num_tr = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        # ("scaler", StandardScaler()) 
    ])
    cat_tr = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ])
    return ColumnTransformer([
        ("num", num_tr, num_cols),
        ("cat", cat_tr, cat_cols)
    ])

rows = []
for ds_name, cfg in CFG.items():
    X, y = load_dataset(ds_name, cfg)

    # SVM
    svm_model = MODELS["SVM (RBF)"]

    # Scaling enabled
    pre_on  = build_preprocessor(X)
    pipe_on = Pipeline([("pre", pre_on), ("clf", svm_model)])
    rows.append(cv_eval_row(pipe_on, X, y, {"dataset": ds_name, "model": "SVM (RBF)", "preproc":"scale_on",  "split":"cv"}))

    # No scaling
    pre_off  = build_preprocessor_no_scaler(X)
    pipe_off = Pipeline([("pre", pre_off), ("clf", svm_model)])
    rows.append(cv_eval_row(pipe_off, X, y, {"dataset": ds_name, "model": "SVM (RBF)", "preproc":"scale_off", "split":"cv"}))

os.makedirs(RESULTS_DIR, exist_ok=True)
pd.DataFrame(rows).to_csv(ABL_OUT, mode="a", header=not ABL_OUT.exists(), index=False)
print(f" Appended {len(rows)} rows to {ABL_OUT}")

 Appended 8 rows to ..\results\ablation.csv


In [12]:
rows = []
for ds_name, cfg in CFG.items():
    X, y = load_dataset(ds_name, cfg)
    pre = build_preprocessor(X)

    for mname, base_model in MODELS.items():
        grid = GRID_BY_MODEL[mname]
        for params in grid:
            pipe = Pipeline([("pre", pre), ("clf", base_model)])
            pipe.set_params(**params)

            meta = {
                "dataset": ds_name,
                "model": mname,
                "preproc": "baseline",
                "split": "cv",
                "params": json.dumps(params, ensure_ascii=False)
            }
            rows.append(cv_eval_row(pipe, X, y, meta, n_splits=5))

os.makedirs(RESULTS_DIR, exist_ok=True)
cv_sweep_path = RESULTS_DIR / "cv_sweep.csv"
pd.DataFrame(rows).to_csv(cv_sweep_path, index=False)
print(f"Saved {len(rows)} CV rows to {cv_sweep_path}")

Saved 52 CV rows to ..\results\cv_sweep.csv


In [13]:
df = pd.read_csv(RESULTS_DIR / "cv_sweep.csv")
(df.sort_values(["dataset","f1_macro_mean"], ascending=[True, False])
   .groupby("dataset")
   .head(5))

,dataset,model,preproc,split,params,cv_splits,acc_mean,acc_std,bal_acc_mean,bal_acc_std,f1_macro_mean,f1_macro_std
17,adult,Random Forest,baseline,cv,"{""clf__n_estimators"": 200, ""clf__max_depth"": 20}",5,0.864685,0.002713,0.775905,0.004359,0.798365,0.004132
19,adult,Random Forest,baseline,cv,"{""clf__n_estimators"": 300, ""clf__max_depth"": 20}",5,0.864531,0.002259,0.775108,0.002424,0.797861,0.002719
18,adult,Random Forest,baseline,cv,"{""clf__n_estimators"": 300, ""clf__max_depth"": n...",5,0.856485,0.002959,0.778778,0.004088,0.793061,0.004206
16,adult,Random Forest,baseline,cv,"{""clf__n_estimators"": 200, ""clf__max_depth"": n...",5,0.856239,0.002421,0.778093,0.003267,0.792545,0.003391
24,adult,SVM (RBF),baseline,cv,"{""clf__C"": 2.0, ""clf__gamma"": ""scale""}",5,0.812352,0.002481,0.830568,0.002894,0.777576,0.002619
26,cancer,Logistic Regression,baseline,cv,"{""clf__C"": 0.1}",5,0.982456,0.011096,0.980592,0.009881,0.981320,0.011706
35,cancer,SVM (RBF),baseline,cv,"{""clf__C"": 1.0, ""clf__gamma"": ""scale""}",5,0.971930,0.021053,0.970411,0.020055,0.970240,0.022151
36,cancer,SVM (RBF),baseline,cv,"{""clf__C"": 1.0, ""clf__gamma"": ""auto""}",5,0.971930,0.021053,0.970411,0.020055,0.970240,0.022151
33,cancer,SVM (RBF),baseline,cv,"{""clf__C"": 0.5, ""clf__gamma"": ""scale""}",5,0.968421,0.023275,0.969538,0.020430,0.966808,0.024218
34,cancer,SVM (RBF),baseline,cv,"{""clf__C"": 0.5, ""clf__gamma"": ""auto""}",5,0.968421,0.023275,0.969538,0.020430,0.966808,0.024218
